# Aplicaciones en Python del artículo **"The FLAMINGO simulations data release"**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Daniel-534/gw-lvk-python-course/blob/main/Articulos/Aplicaciones_FLAMINGO_Data_Release.ipynb)

Este notebook desarrolla, de forma práctica y ejecutable en **Google Colab**, aplicaciones inspiradas en el artículo del *data release* de las simulaciones cosmológicas FLAMINGO (Helly et al., 2026).

> Objetivo: traducir los conceptos del artículo (simulaciones hidrodinámicas, productos de datos masivos, catálogos, lightcones y espectros de potencia) a un flujo de trabajo reproducible en Python para análisis científico.

## 1. ¿Qué aporta FLAMINGO y por qué importa?

Según el artículo, FLAMINGO libera más de **2.3 PB** de datos de simulaciones cosmológicas de gran volumen, incluyendo:

- snapshots de partículas (HDF5),
- catálogos de halos y galaxias,
- mapas all-sky tipo HEALPix de lightcones,
- partículas en lightcones,
- espectros de potencia 3D y cruzados.

También enfatiza dos ideas clave para cosmología moderna:

1. **Los efectos bariónicos importan** en escalas no lineales (impactan lente débil, tSZ, clustering).
2. **El acceso remoto selectivo** es necesario, porque descargar todo el dataset no es realista para la mayoría de usuarios.

En este notebook construiremos mini-ejemplos que reflejan exactamente esos dos mensajes.

In [ ]:
# Instalación robusta para Colab/entornos limpios
import importlib, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'scipy', 'h5py']
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]

if missing:
    print('Instalando paquetes faltantes:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing)
else:
    print('Dependencias principales ya disponibles ✅')


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.interpolate import interp1d
import h5py
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
rng = np.random.default_rng(12345)

print('Entorno listo ✅')


## 2. Resumen técnico del artículo en formato de trabajo

Convertimos la información cualitativa del paper a una tabla de referencia útil para análisis en Python.

In [ ]:
flamingo_release = {
    'data_volume_pb': 2.3,
    'key_products': [
        'Snapshots HDF5',
        'Catálogos de halos/galaxias (SOAP/HBT+)',
        'Mapas all-sky de lightcone',
        'Partículas de lightcone',
        'Espectros de potencia 3D y cruzados'
    ],
    'science_axes': [
        'Efectos bariónicos en P(k)',
        'Variaciones cosmológicas (incluyendo neutrinos masivos)',
        'Predicción de observables de gran escala',
        'Calibración de modelos subgrid'
    ],
    'access_model': 'Descarga total + acceso remoto selectivo por web service'
}

summary_df = pd.DataFrame({
    'Componente': ['Volumen total', 'Productos principales', 'Ejes científicos', 'Modelo de acceso'],
    'Detalle': [
        f"{flamingo_release['data_volume_pb']} PB",
        '; '.join(flamingo_release['key_products']),
        '; '.join(flamingo_release['science_axes']),
        flamingo_release['access_model']
    ]
})
summary_df


## 3. Diseño de un mini-laboratorio reproducible

Como no descargaremos terabytes, generaremos datasets **toy pero físicamente plausibles** para practicar:

- catálogos de halos con masa y redshift,
- espectros de potencia con supresión bariónica/neutrinos,
- lightcone simplificado para mapas y conteos,
- snapshots HDF5 sintéticos con extracción regional.

Este enfoque permite **enseñar y validar metodología** en Colab antes de escalar a FLAMINGO real.

## 4. Generación de catálogos sintéticos tipo FLAMINGO

Simularemos tres escenarios para comparar:

- **DMO** (solo materia oscura),
- **Hydro-fiducial** (hidrodinámico calibrado),
- **Hydro + neutrinos** (supresión extra en estructuras pequeñas).

In [ ]:
def sample_halo_masses(n, logm_min=11.0, logm_max=15.2, alpha=1.85, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    u = rng.uniform(size=n)
    a = 1 - alpha
    x1, x2 = 10**logm_min, 10**logm_max
    masses = (u * (x2**a - x1**a) + x1**a) ** (1/a)
    return masses


def synthetic_catalog(n=250_000, model='DMO', rng=None):
    rng = np.random.default_rng() if rng is None else rng
    z = rng.uniform(0, 2.0, size=n)

    if model == 'DMO':
        masses = sample_halo_masses(n, alpha=1.80, rng=rng)
        gas_frac = np.zeros(n)
    elif model == 'Hydro-fiducial':
        masses = sample_halo_masses(n, alpha=1.90, rng=rng)
        gas_frac = np.clip(0.08 + 0.03*np.log10(masses/1e13) + rng.normal(0, 0.01, n), 0, 0.2)
    elif model == 'Hydro+nu':
        masses = sample_halo_masses(n, alpha=1.98, rng=rng)
        gas_frac = np.clip(0.075 + 0.028*np.log10(masses/1e13) + rng.normal(0, 0.012, n), 0, 0.2)
    else:
        raise ValueError('Modelo no reconocido')

    # proxy simple de formación estelar específica decreciente con masa
    sSFR = np.clip(1e-10 * (masses/1e12)**(-0.35) * (1+z)**1.2 * np.exp(rng.normal(0, 0.35, n)), 1e-13, 3e-9)

    return pd.DataFrame({'z': z, 'M_halo': masses, 'f_gas': gas_frac, 'sSFR': sSFR, 'model': model})

cat_dmo = synthetic_catalog(model='DMO', rng=rng)
cat_hydro = synthetic_catalog(model='Hydro-fiducial', rng=rng)
cat_nu = synthetic_catalog(model='Hydro+nu', rng=rng)

catalog = pd.concat([cat_dmo, cat_hydro, cat_nu], ignore_index=True)
catalog.head()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for model, c in [('DMO', '#1f77b4'), ('Hydro-fiducial', '#ff7f0e'), ('Hydro+nu', '#2ca02c')]:
    data = catalog.loc[catalog.model == model, 'M_halo']
    axes[0].hist(np.log10(data), bins=60, alpha=0.55, density=True, color=c, label=model)

axes[0].set_xlabel('log10(M_halo / Msun)')
axes[0].set_ylabel('Densidad')
axes[0].set_title('Distribución de masas de halo')
axes[0].legend()

for model, c in [('Hydro-fiducial', '#ff7f0e'), ('Hydro+nu', '#2ca02c')]:
    data = catalog[catalog.model == model]
    axes[1].scatter(np.log10(data['M_halo'].sample(20000, random_state=1)),
                    data['f_gas'].sample(20000, random_state=1),
                    s=1, alpha=0.15, color=c, label=model)

axes[1].set_xlabel('log10(M_halo / Msun)')
axes[1].set_ylabel('f_gas')
axes[1].set_title('Fracción de gas (proxy)')
axes[1].legend(markerscale=6)

for model, c in [('DMO', '#1f77b4'), ('Hydro-fiducial', '#ff7f0e'), ('Hydro+nu', '#2ca02c')]:
    data = catalog[catalog.model == model].sample(20000, random_state=2)
    axes[2].scatter(data['z'], np.log10(data['sSFR']), s=1, alpha=0.15, color=c, label=model)

axes[2].set_xlabel('z')
axes[2].set_ylabel('log10(sSFR / yr^-1)')
axes[2].set_title('Evolución de actividad estelar (proxy)')
axes[2].legend(markerscale=6)

plt.tight_layout()
plt.show()


## 5. Aplicación 1: función de masa de halos

Una aplicación directa de catálogos FLAMINGO es medir cómo cambian abundancias de halos según física bariónica/cosmología.

In [ ]:
def halo_mass_function(df, zmin=0.0, zmax=0.5, bins=np.linspace(11, 15.2, 34)):
    d = df[(df.z >= zmin) & (df.z < zmax)]
    h, edges = np.histogram(np.log10(d.M_halo), bins=bins)
    centers = 0.5*(edges[1:] + edges[:-1])
    phi = h / np.diff(edges)  # conteos por bin logM
    return centers, phi

bins = np.linspace(11, 15.2, 34)
fig, ax = plt.subplots(figsize=(9, 5))

curves = {}
for model, c in [('DMO', '#1f77b4'), ('Hydro-fiducial', '#ff7f0e'), ('Hydro+nu', '#2ca02c')]:
    cc, phi = halo_mass_function(catalog[catalog.model == model], zmin=0.0, zmax=0.5, bins=bins)
    curves[model] = (cc, phi)
    ax.plot(cc, phi + 1e-9, lw=2, color=c, label=model)

ax.set_yscale('log')
ax.set_xlabel('log10(M_halo / Msun)')
ax.set_ylabel('N / dlogM (arbitrario)')
ax.set_title('Comparación de función de masa de halos')
ax.legend()
plt.show()

# Ratio respecto a DMO
cc = curves['DMO'][0]
ratio_hydro = curves['Hydro-fiducial'][1] / np.maximum(curves['DMO'][1], 1)
ratio_nu = curves['Hydro+nu'][1] / np.maximum(curves['DMO'][1], 1)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(cc, ratio_hydro, lw=2, label='Hydro / DMO')
ax.plot(cc, ratio_nu, lw=2, label='Hydro+nu / DMO')
ax.axhline(1.0, color='k', ls='--', lw=1)
ax.set_xlabel('log10(M_halo / Msun)')
ax.set_ylabel('Ratio de abundancia')
ax.set_title('Impacto relativo de física adicional')
ax.legend()
plt.show()


### Interpretación

- En este ejemplo, los modelos hidrodinámicos reducen halos de menor masa efectiva (feedback).
- El escenario con neutrinos muestra supresión adicional.
- Este tipo de comparación conecta con los objetivos del artículo: cuantificar variaciones de física y cosmología de forma sistemática.

## 6. Aplicación 2: espectro de potencia y supresiones relativas

FLAMINGO publica espectros de potencia 3D (materia total, CDM, gas, neutrinos, etc.).
Aquí implementamos un modelo analítico simple para entrenar el análisis de:

\[ S(k) = P_{\mathrm{modelo}}(k)/P_{\mathrm{DMO}}(k) \]

que es exactamente la razón usada en muchos estudios para aislar efectos físicos.

In [ ]:
k = np.logspace(-2.2, 1.2, 240)  # h/Mpc

# P(k) base (toy)
P_dmo = 4e3 * (k/0.2)**(-1.8) / (1 + (k/0.35)**2.2)

# Supresión bariónica (toy): más fuerte en k intermedio/alto
S_bary = 1 - 0.12 * (1 - np.exp(-(k/0.7)**1.3))

# Supresión adicional por neutrinos (toy)
S_nu = 1 - 0.06 * (1 - np.exp(-(k/0.35)**1.1))

P_hydro = P_dmo * S_bary
P_hydro_nu = P_hydro * S_nu

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].loglog(k, P_dmo, lw=2, label='DMO')
axes[0].loglog(k, P_hydro, lw=2, label='Hydro-fiducial')
axes[0].loglog(k, P_hydro_nu, lw=2, label='Hydro+nu')
axes[0].set_xlabel('k [h/Mpc]')
axes[0].set_ylabel('P(k) [arb.]')
axes[0].set_title('Espectros de potencia (toy)')
axes[0].legend()

axes[1].semilogx(k, P_hydro / P_dmo, lw=2, label='Hydro / DMO')
axes[1].semilogx(k, P_hydro_nu / P_dmo, lw=2, label='Hydro+nu / DMO')
axes[1].axhline(1, color='k', ls='--', lw=1)
axes[1].set_xlabel('k [h/Mpc]')
axes[1].set_ylabel('Ratio')
axes[1].set_title('Supresión relativa por física adicional')
axes[1].legend()

plt.tight_layout()
plt.show()


## 7. Aplicación 3: impacto en un observable de lente débil (aproximado)

Con una proyección simplificada podemos estimar cómo una supresión en P(k) desplaza un pseudo-$C_\ell$ integrado.

In [ ]:
ell = np.arange(50, 3001)
# Relación aproximada k ~ (ell+0.5)/chi_eff
chi_eff = 1800.0  # Mpc/h (toy)
k_ell = (ell + 0.5) / chi_eff

interp_dmo = interp1d(k, P_dmo, bounds_error=False, fill_value='extrapolate')
interp_h = interp1d(k, P_hydro, bounds_error=False, fill_value='extrapolate')
interp_hn = interp1d(k, P_hydro_nu, bounds_error=False, fill_value='extrapolate')

# Kernel efectivo toy W(z)^2 integrado -> factor de normalización
norm = 1e-8
C_dmo = norm * interp_dmo(k_ell)
C_h = norm * interp_h(k_ell)
C_hn = norm * interp_hn(k_ell)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].loglog(ell, C_dmo, lw=2, label='DMO')
axes[0].loglog(ell, C_h, lw=2, label='Hydro-fiducial')
axes[0].loglog(ell, C_hn, lw=2, label='Hydro+nu')
axes[0].set_xlabel(r'$\ell$')
axes[0].set_ylabel(r'$C_\ell$ [arb.]')
axes[0].set_title('Pseudo espectro angular de lente débil')
axes[0].legend()

axes[1].semilogx(ell, C_h/C_dmo, lw=2, label='Hydro / DMO')
axes[1].semilogx(ell, C_hn/C_dmo, lw=2, label='Hydro+nu / DMO')
axes[1].axhline(1, color='k', ls='--', lw=1)
axes[1].set_xlabel(r'$\ell$')
axes[1].set_ylabel('Ratio')
axes[1].set_title('Cambio relativo en observable proyectado')
axes[1].legend()

plt.tight_layout()
plt.show()


## 8. Aplicación 4: lightcone simplificado y mapa all-sky

El artículo destaca productos all-sky tipo HEALPix.
Aquí construimos una versión simplificada (sin dependencia de healpy) para practicar lógica de lightcone y visualización en esfera.

In [ ]:
n_obj = 300_000
ra = rng.uniform(-np.pi, np.pi, n_obj)      # rad
u = rng.uniform(-1, 1, n_obj)
dec = np.arcsin(u)                           # rad
z = rng.uniform(0, 3.0, n_obj)

# peso toy (ejemplo: señal integrada que crece con z y masa efectiva)
mass_proxy = 10**rng.uniform(11, 14.5, n_obj)
signal = (mass_proxy/1e12)**0.25 * (1 + z)**0.8

# pixelización simple en malla regular RA/Dec
n_ra, n_dec = 240, 120
ra_edges = np.linspace(-np.pi, np.pi, n_ra+1)
dec_edges = np.linspace(-np.pi/2, np.pi/2, n_dec+1)

H, _, _ = np.histogram2d(dec, ra, bins=[dec_edges, ra_edges], weights=signal)
N, _, _ = np.histogram2d(dec, ra, bins=[dec_edges, ra_edges])
mean_map = H / np.maximum(N, 1)

fig = plt.figure(figsize=(12, 5))
ax = fig.add_subplot(111, projection='mollweide')
ra_cent = 0.5*(ra_edges[:-1]+ra_edges[1:])
dec_cent = 0.5*(dec_edges[:-1]+dec_edges[1:])
RA, DEC = np.meshgrid(ra_cent, dec_cent)

im = ax.pcolormesh(RA, DEC, np.log10(mean_map + 1e-3), shading='auto', cmap='magma')
ax.grid(True, alpha=0.4)
ax.set_title('Mapa all-sky simplificado (log10 señal media por píxel)')
cb = plt.colorbar(im, ax=ax, pad=0.08)
cb.set_label('log10(señal proxy)')
plt.show()


## 9. Aplicación 5: snapshots HDF5 y extracción por región

En FLAMINGO real, el artículo propone acceso selectivo y lectura regional para evitar descargas masivas.
Aquí replicamos ese patrón con un archivo HDF5 pequeño local.

In [ ]:
toy_file = Path('/tmp/flamingo_toy_snapshot.hdf5')
N = 800_000

coords = rng.uniform(0, 1000, size=(N, 3)).astype('f4')  # Mpc comoving
density = (10**rng.uniform(-2, 2, size=N)).astype('f4')
temp = (10**rng.uniform(4, 8, size=N)).astype('f4')

with h5py.File(toy_file, 'w') as f:
    g = f.create_group('gas')
    dset_c = g.create_dataset('coordinates', data=coords, compression='gzip')
    dset_d = g.create_dataset('density', data=density, compression='gzip')
    dset_t = g.create_dataset('temperature', data=temp, compression='gzip')

    dset_c.attrs['units'] = 'Mpc (comoving)'
    dset_d.attrs['units'] = 'arbitrary'
    dset_t.attrs['units'] = 'K'

print(f'Archivo creado: {toy_file} | tamaño ~ {toy_file.stat().st_size/1e6:.1f} MB')


In [ ]:
# Extracción de una región cúbica de interés
region = np.array([[200, 350], [400, 550], [100, 250]], dtype=float)

with h5py.File(toy_file, 'r') as f:
    c = f['gas/coordinates'][...]
    d = f['gas/density'][...]
    t = f['gas/temperature'][...]

mask = (
    (c[:,0] >= region[0,0]) & (c[:,0] < region[0,1]) &
    (c[:,1] >= region[1,0]) & (c[:,1] < region[1,1]) &
    (c[:,2] >= region[2,0]) & (c[:,2] < region[2,1])
)

sub = pd.DataFrame({
    'x': c[mask,0],
    'y': c[mask,1],
    'z': c[mask,2],
    'density': d[mask],
    'temperature': t[mask]
})

print(f'Partículas totales: {len(c):,}')
print(f'Partículas en región: {len(sub):,}')
sub.describe().T[['mean','std','min','max']]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(np.log10(sub['density']), bins=50, color='#1f77b4', alpha=0.8)
axes[0].set_xlabel('log10(density)')
axes[0].set_ylabel('Conteo')
axes[0].set_title('Distribución de densidad en región extraída')

axes[1].hist(np.log10(sub['temperature']), bins=50, color='#d62728', alpha=0.8)
axes[1].set_xlabel('log10(T [K])')
axes[1].set_ylabel('Conteo')
axes[1].set_title('Distribución de temperatura en región extraída')

plt.tight_layout()
plt.show()


## 10. Aplicación 6: mini-pipeline científico de extremo a extremo

Integramos en una sola rutina:

1. selección de muestra por redshift,
2. estimación de función de masa,
3. comparación de modelos,
4. cálculo de métrica de discrepancia simple (KS) para distribuciones de masa.

In [ ]:
def end_to_end_report(catalog):
    out = {}
    zslice = (0.2, 0.6)
    bins = np.linspace(11, 15.2, 28)

    models = ['DMO', 'Hydro-fiducial', 'Hydro+nu']
    for m in models:
        dfm = catalog[(catalog.model == m) & (catalog.z >= zslice[0]) & (catalog.z < zslice[1])]
        hist, edges = np.histogram(np.log10(dfm.M_halo), bins=bins)
        out[m] = {
            'N': len(dfm),
            'hist': hist,
            'centers': 0.5*(edges[1:]+edges[:-1]),
            'median_logM': np.median(np.log10(dfm.M_halo))
        }

    dmo = catalog[(catalog.model == 'DMO') & (catalog.z >= zslice[0]) & (catalog.z < zslice[1])]
    hydro = catalog[(catalog.model == 'Hydro-fiducial') & (catalog.z >= zslice[0]) & (catalog.z < zslice[1])]
    hnu = catalog[(catalog.model == 'Hydro+nu') & (catalog.z >= zslice[0]) & (catalog.z < zslice[1])]

    ks_h = stats.ks_2samp(np.log10(dmo.M_halo), np.log10(hydro.M_halo))
    ks_hn = stats.ks_2samp(np.log10(dmo.M_halo), np.log10(hnu.M_halo))

    return out, ks_h, ks_hn

report, ks_h, ks_hn = end_to_end_report(catalog)

print('Resumen del pipeline (z en [0.2,0.6]):')
for m, res in report.items():
    print(f"- {m:15s} | N={res['N']:,} | mediana logM={res['median_logM']:.3f}")

print() 
print('KS tests vs DMO:')
print(f"Hydro-fiducial: statistic={ks_h.statistic:.4f}, p={ks_h.pvalue:.2e}")
print(f"Hydro+nu       : statistic={ks_hn.statistic:.4f}, p={ks_hn.pvalue:.2e}")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for m, c in [('DMO', '#1f77b4'), ('Hydro-fiducial', '#ff7f0e'), ('Hydro+nu', '#2ca02c')]:
    ax.plot(report[m]['centers'], report[m]['hist'] + 1e-6, lw=2, color=c, label=m)

ax.set_yscale('log')
ax.set_xlabel('log10(M_halo / Msun)')
ax.set_ylabel('Conteo por bin')
ax.set_title('Mini-pipeline: comparación final de funciones de masa')
ax.legend()
plt.show()


## 11. Conexión con el acceso remoto real descrito en el artículo

El paper documenta un servicio remoto y ejemplos en Python (`hdfstream` + `swiftsimio`) para abrir archivos HDF5 sin descarga total.

La celda siguiente deja un **template ejecutable** (se ignora si el servicio no está accesible desde tu entorno).

In [ ]:
# Template opcional basado en el Apéndice A del artículo
# Se ejecuta solo si instalas dependencias y tienes conectividad al servicio.

try:
    import importlib
    hdfstream_spec = importlib.util.find_spec('hdfstream')
    sw_spec = importlib.util.find_spec('swiftsimio')

    if hdfstream_spec is None or sw_spec is None:
        print('Dependencias opcionales no instaladas. Si quieres, ejecuta:')
        print('!pip -q install hdfstream swiftsimio unyt')
    else:
        import hdfstream
        import swiftsimio as sw
        print('Paquetes disponibles. Puedes probar acceso remoto si tu entorno tiene permisos/red.')

        # Ejemplo (comentado para evitar fallo automático):
        # root_dir = hdfstream.open('cosma', '/')
        # snap_file = root_dir['FLAMINGO/L1_m10/L1_m10/snapshots/flamingo_0077/flamingo_0077.hdf5']
        # snap = sw.load(snap_file)
        # star_pos = snap.stars.coordinates

except Exception as e:
    print('Sección opcional omitida por entorno:', e)


## 12. Casos de uso reales que puedes construir a partir de FLAMINGO

1. **Calibración de modelos de contaminación bariónica** en análisis de lente débil (ratios en P(k), Cℓ).
2. **Pruebas de sensibilidad a masa de neutrino** en observables de estructura a gran escala.
3. **Entrenamiento de emuladores** (ML/surrogates) para acelerar inferencia cosmológica.
4. **Catálogos sintéticos para diseño de surveys** (selección, completitud, efectos de proyección).
5. **Cross-correlations multi-trazador** (galaxias, tSZ, X-ray, CIB, FRB DM) usando mapas de lightcone.
6. **Validación de pipelines de análisis regional** (lectura parcial de snapshots para reducir costo de I/O).

## 13. Posibles investigaciones futuras alrededor de esta información

### A. Cosmología y parámetros fundamentales
- Inferencia conjunta de $\sigma_8$, $\Omega_m$, $\sum m_
u$ incluyendo incertidumbre bariónica explícita.
- Comparación de tensiones entre CMB y low-z con escenarios de feedback alternativo.

### B. Astrofísica de baryones
- Sensibilidad del perfil gas/temperatura de halos a variaciones de AGN feedback.
- Relación halo-galaxia y su evolución en volumen cosmológico grande.

### C. Metodología computacional
- Estrategias óptimas de *remote slicing* para análisis reproducible de datasets PB-scale.
- Compresión de datos y aprendizaje auto-supervisado sobre snapshots y lightcones.

### D. Ciencia multi-mensajero y de fondo difuso
- Predicciones conjuntas para tSZ/kSZ + X-ray + FRB DM + clustering galáctico.
- Uso de mapas integrados para optimizar campañas observacionales futuras.

## 14. Conclusiones

Este notebook tradujo el artículo de FLAMINGO a un entorno práctico en Python/Colab mediante:

- generación y análisis de catálogos tipo FLAMINGO,
- medición de supresiones en espectros de potencia,
- proyecciones a observables,
- construcción de lightcones simplificados,
- extracción regional de snapshots HDF5,
- y diseño de rutas de investigación realistas.

Con este flujo, puedes pasar de pruebas didácticas reproducibles a análisis sobre productos reales del data release.

---
### Referencia principal
Helly, J. C., et al. (2026). *The FLAMINGO simulations data release*. arXiv:2604.24324.

### Nota
Los datos numéricos de este notebook son sintéticos para fines pedagógicos y de prototipado metodológico; la lógica de análisis está diseñada para migrar fácilmente a datos reales FLAMINGO.